# Spanish vessel statistics by port and year

## Purpose

This notebook prepares the annual vessel-count dataset used by the EFFKG
Wikibase population pipeline to represent the number of registered fishing
vessels associated with each Spanish base port over time.

The source workbook contains one row per active vessel and year. The notebook
aggregates these records by base port and observation year and exports the
result as a compact CSV consumed by `import_wikibase.ipynb`.

## Processing workflow

1. Load the RGFP Excel workbook.
2. Validate the required source columns.
3. Normalize base-port names.
4. Remove incomplete records.
5. Count active vessels by base port and year.
6. Export the aggregated dataset.

## Output

The generated file:

`source_data/processed/stats/vessels_year_port.csv`

contains one row per:

- base port;
- observation year.

The notebook prepares data only and does not interact with Wikibase.

# Setup

In [ ]:
# =============================================================================
# SETUP AND CONFIGURATION
# =============================================================================

from pathlib import Path
from typing import Optional

import pandas as pd


def find_repository_root(start: Optional[Path] = None) -> Path:
    """
    Locate the EFFKG repository root.

    The root is identified by the principal directories distributed with the
    project. This implementation is compatible with Python 3.9.
    """
    current = (start or Path.cwd()).resolve()

    required_directories = {
        "code",
        "dataset",
        "schema",
        "source_data",
        "data_model",
        "validation",
    }

    for candidate in [current] + list(current.parents):
        try:
            existing_directories = {
                path.name
                for path in candidate.iterdir()
                if path.is_dir()
            }
        except (PermissionError, OSError):
            continue

        if required_directories.issubset(existing_directories):
            return candidate

    raise RuntimeError(
        "The EFFKG repository root could not be located. "
        "Run this notebook from within a cloned EFFKG repository."
    )


REPOSITORY_ROOT = find_repository_root()

SOURCE_DATA_DIRECTORY = (
    REPOSITORY_ROOT
    / "source_data"
)

PROCESSED_STATS_DIRECTORY = (
    SOURCE_DATA_DIRECTORY
    / "processed"
    / "stats"
)

PROCESSED_STATS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# -----------------------------------------------------------------------------
# Input and output files
# -----------------------------------------------------------------------------

INPUT_FILE = (
    SOURCE_DATA_DIRECTORY
    / "rgfp_bi_excel_2006_2025_20260127.xlsx"
)

OUTPUT_FILE = (
    PROCESSED_STATS_DIRECTORY
    / "vessels_year_port.csv"
)

SHEET_NAME = "Datos"

REQUIRED_COLUMNS = [
    "AÑO",
    "CODIGOBUQUE",
    "PUERTO BASE",
]


# -----------------------------------------------------------------------------
# Input validation
# -----------------------------------------------------------------------------

if not INPUT_FILE.is_file():
    raise FileNotFoundError(
        "The RGFP input workbook was not found:\n"
        "  {}\n\n"
        "Place the workbook in source_data/ or update INPUT_FILE in the "
        "configuration cell.".format(INPUT_FILE)
    )


print("EFFKG Spanish vessel statistics preparation")
print("--------------------------------------------")
print("Repository root: {}".format(REPOSITORY_ROOT))
print("Input workbook: {}".format(INPUT_FILE))
print("Worksheet: {}".format(SHEET_NAME))
print("Output CSV: {}".format(OUTPUT_FILE))

# Helpers

In [6]:
def load_rgfp_excel(path, sheet_name):
    """
    Load the configured RGFP Excel workbook into a pandas dataframe.
    """
    return pd.read_excel(path, sheet_name=sheet_name)


def validate_columns(df, required_columns):
    """
    Ensure the input dataframe contains the columns required by the aggregation.
    """
    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}. "
            f"Available columns: {list(df.columns)}"
        )


def normalize_port_name(value):
    """
    Normalize a base-port name.

    The normalization:

    - removes leading and trailing whitespace;
    - collapses repeated internal whitespace;
    - converts empty values to missing values.
    """
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()
    text = " ".join(text.split())

    return text if text else pd.NA


def prepare_vessel_rows(df):
    """
    Prepare the source records used for aggregation.

    The function:

    - validates the required source columns;
    - keeps only the fields required by the aggregation;
    - normalizes base-port names;
    - removes incomplete records.

    Each remaining row represents one active vessel during one observation year.
    """
    validate_columns(df, REQUIRED_COLUMNS)

    prepared = df[REQUIRED_COLUMNS].copy()

    prepared["PUERTO BASE"] = prepared["PUERTO BASE"].apply(normalize_port_name)

    # Rows without a year or a base port cannot be aggregated reliably.
    prepared = prepared.dropna(subset=["AÑO", "PUERTO BASE"])

    return prepared


def aggregate_boats_by_port_year(df):
    """
    Aggregate active vessels by base port and observation year.

    The input workbook is assumed to contain one row per active vessel-year
    combination.
    """
    counts = (
        df.groupby(["PUERTO BASE", "AÑO"], as_index=False)
          .size()
          .rename(columns={"size": "NUM_BARCOS"})
    )

    return counts


def save_output(df, path):
    """
    Export the aggregated statistics as a UTF-8 semicolon-separated CSV.
    """
    df.to_csv(path, sep=";", index=False, encoding="utf-8-sig")


# Pipeline execution

In [ ]:
df = load_rgfp_excel(INPUT_FILE, SHEET_NAME)
df = prepare_vessel_rows(df)
counts_long = aggregate_boats_by_port_year(df)

save_output(counts_long, OUTPUT_FILE)

print("Output generated in:")
print(OUTPUT_FILE)
print(f"Rows exported: {len(counts_long)}")